<a href="https://colab.research.google.com/github/rwcitek/TechEx-LangGraph-workshop/blob/main/notebooks/solutions/04_observability_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4 — Observability in Practice

> **Time:** 20 minutes.
>
> **What you'll do:** wire LangSmith into the system you fixed in Module 3, attach searchable metadata, and find a planted slow node — all from the trace dashboard.

By the end of this notebook you'll be able to answer questions like *"which ticket category is slowest?"* and *"why was ticket T-1247 slow?"* from the traces alone — no print statements required.

**Prerequisite:** a `LANGSMITH_API_KEY` (free at https://smith.langchain.com). If you don't have one, you can still run the notebook — you just won't see the dashboard view.

> *Solution notebook. Try the starter first.*

## 1.  Setup

Three env vars and zero code changes — that's the LangSmith promise. After this cell, every LangChain/LangGraph call is captured.

In [1]:
%pip install -q \
    langgraph==0.2.* \
    langchain==0.3.* \
    langchain-openai==0.2.* \
    langsmith==0.1.*


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.63 which is incompatibl

In [2]:
import os
from langchain_openai import ChatOpenAI
import json
from typing import TypedDict, Annotated, Literal
from operator import add
from langgraph.graph import StateGraph, END
import time


In [3]:
def _ensure_key(name: str, optional: bool = False) -> None:
    """Load an API key from (in order): existing env, Colab Secrets, or getpass.

    Colab Secrets are the recommended path for this workshop — set them ONCE
    via the 🔑 key icon in Colab's left sidebar and every notebook will pick
    them up automatically.
    """
    if os.environ.get(name):
        print(f"  ✓  {name} already set in environment")
        return
    # 1. Try Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✓  {name} loaded from Colab Secrets")
            return
    except Exception:
        pass
    # 2. Fallback — prompt the user
    import getpass
    val = getpass.getpass(f"Paste your {name}{' (optional)' if optional else ''}: ").strip()
    if val:
        os.environ[name] = val
        print(f"  ✓  {name} set")
    elif optional:
        print(f"  •  {name} skipped (optional)")
    else:
        print(f"  ⚠   {name} skipped — you'll hit errors later without it")

_ensure_key("OPENAI_API_KEY")
_ensure_key("LANGSMITH_API_KEY", optional=True)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"]    = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]     = "triage-workshop"

print("Project           →", os.environ.get("LANGCHAIN_PROJECT"))


  ✓  OPENAI_API_KEY loaded from Colab Secrets
  ✓  LANGSMITH_API_KEY loaded from Colab Secrets
Project           → triage-workshop


## 2.  KB + state schema

In [4]:
KB = {
    "billing": [
        {"id": "B-001", "title": "Billing cycle and prorations",
         "text": "We bill on the same day each month. If you change plans mid-cycle, the next invoice is prorated."},
        {"id": "B-002", "title": "Refund policy",
         "text": "Refunds are available within 14 days. Issued to original payment method, settle in 5 business days."},
        {"id": "B-003", "title": "Failed payments",
         "text": "Failed payments retry once a day for 3 days, then 7-day grace period."},
    ],
    "technical": [
        {"id": "T-001", "title": "Login issues",
         "text": "Clear cookies. For MFA failures check spam and verify phone."},
        {"id": "T-002", "title": "API rate limits",
         "text": "Standard plan: 60 req/min. Enterprise: 600. Respect Retry-After."},
        {"id": "T-003", "title": "Export failures",
         "text": "Retry from the same dialog — export jobs are idempotent."},
    ],
    "account": [
        {"id": "A-001", "title": "Password reset",
         "text": "Use Forgot password. Reset email valid 30 minutes. Check spam."},
        {"id": "A-002", "title": "Account closure",
         "text": "Settings → Account → Close. 30-day pending-deletion window."},
        {"id": "A-003", "title": "Ownership transfer",
         "text": "Owner invites new owner as Admin, both confirm via email."},
    ],
}

class TriageState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "account"] | None
    urgency:  Literal["low", "med", "high"] | None
    retrieved: list[dict]
    draft: str
    verdict: Literal["pass", "revise"] | None
    revision_count: int
    revisions: Annotated[list[str], add]


def make_initial_state(ticket: str) -> TriageState:
    return {"ticket": ticket, "category": None, "urgency": None,
            "retrieved": [], "draft": "", "verdict": None,
            "revision_count": 0, "revisions": []}

## 3.  Agents and graph

In [5]:
# Cheap, fast model for most nodes.
llm_fast = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 🐢  Planted regression: Drafter uses gpt-4o, which is significantly slower.
# This is the bottleneck you'll find in Step 3.
llm_drafter = ChatOpenAI(model="gpt-4o", temperature=0)


def parse_json(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"): text = text[4:]
    return json.loads(text.strip())


CLASSIFY_PROMPT = """\
Categorize the ticket as one of: billing, technical, account.
Rate urgency as: low, med, high.
Return JSON: {{"category": "...", "urgency": "..."}}

TICKET: {ticket}
"""

def classify(state: TriageState) -> dict:
    prompt = CLASSIFY_PROMPT.format(ticket=state["ticket"])
    response = llm_fast.invoke(prompt)
    parsed = parse_json(response.content)
    return {"category": parsed["category"], "urgency": parsed["urgency"]}


def retrieve(state: TriageState) -> dict:
    return {"retrieved": KB.get(state["category"], [])}


DRAFT_PROMPT = """\
You are a customer support agent.

Reply to the ticket using ONLY the policies below. If a policy doesn't
answer the question, say so. Do not invent details.

POLICIES:
{context}

TICKET:
{ticket}

Write a concise reply (3-6 sentences).
"""

def draft(state: TriageState) -> dict:
    context = "\n\n".join(f"[{d['id']}] {d['title']}\n{d['text']}"
                            for d in state["retrieved"])
    prompt = DRAFT_PROMPT.format(context=context, ticket=state["ticket"])
    # Note: this uses the SLOWER LLM
    response = llm_drafter.invoke(prompt)
    return {"draft": response.content, "revisions": [response.content]}


QA_PROMPT = """\
Decide if the DRAFT is acceptable. Return JSON: {{"verdict": "pass" | "revise"}}

TICKET: {ticket}
DRAFT:  {draft}
"""

def qa(state: TriageState) -> dict:
    prompt = QA_PROMPT.format(ticket=state["ticket"], draft=state["draft"])
    response = llm_fast.invoke(prompt)
    parsed = parse_json(response.content)
    return {"verdict": parsed["verdict"],
            "revision_count": state["revision_count"] + 1}

In [6]:
MAX_REVISIONS = 2

def route_qa(state: TriageState) -> str:
    if state["verdict"] == "pass":     return "END"
    if state["revision_count"] >= MAX_REVISIONS: return "END"
    return "drafter"

graph = StateGraph(TriageState)
graph.add_node("classify", classify)
graph.add_node("retrieve", retrieve)
graph.add_node("drafter",  draft)
graph.add_node("qa",       qa)

graph.set_entry_point("classify")
graph.add_edge("classify", "retrieve")
graph.add_edge("retrieve", "drafter")
graph.add_edge("drafter",  "qa")
graph.add_conditional_edges("qa", route_qa,
    {"drafter": "drafter", "END": END})

app = graph.compile()
print("Graph compiled.")

Graph compiled.


## 4.  Step 1 — Wire up LangSmith and run 5 tickets

Now run 5 tickets. Because tracing is on, every run shows up in your LangSmith dashboard under the `triage-workshop` project.

Open https://smith.langchain.com → your project → confirm you see 5 new runs.

In [7]:
sample_tickets = [
    "Hi, my monthly bill is $50 higher than last month. Can you check what happened?",
    "I keep getting 429 errors from your API on the Standard plan.",
    "Forgot my password and the reset email never showed up. Demo in 30 minutes!",
    "I want to close my account. Will my data really be deleted?",
    "Charged me twice for the same month. Refund please.",
]

for t in sample_tickets:
    t0 = time.time()
    result = app.invoke(make_initial_state(t))
    elapsed = time.time() - t0
    print(f"  {elapsed:5.2f}s   {result['category']:9}   {t[:55]}...")

   4.32s   billing     Hi, my monthly bill is $50 higher than last month. Can ...
   2.03s   technical   I keep getting 429 errors from your API on the Standard...
   9.11s   account     Forgot my password and the reset email never showed up....
   3.13s   account     I want to close my account. Will my data really be dele...
   2.21s   billing     Charged me twice for the same month. Refund please....


## 5.  Step 2 — Attach searchable metadata

Right now the traces are visible but anonymous — you can't easily filter by category or ticket ID.

LangGraph (via LangChain Runnables) accepts a `config` argument with `tags` and `metadata`. These show up on the run and are searchable in the dashboard.

**Your task:** wrap the same 5 tickets, but this time pass `tags` and `metadata`. Then, in the dashboard, filter the runs view by `metadata.category == "billing"`.

In [8]:
for i, t in enumerate(sample_tickets):
    # Do a quick local classify so we know how to tag the run
    # (in production you'd do this in a metadata service or upstream)
    local_category = "billing" if "bill" in t.lower() or "refund" in t.lower() or "charged" in t.lower() \
                    else "technical" if "API" in t or "export" in t.lower() or "login" in t.lower() \
                    else "account"
    urgent = "urgent" in t.lower() or "demo" in t.lower() or "!" in t

    tags = [
        f"category:{local_category}",
        "urgent" if urgent else "non-urgent",
    ]
    metadata = {
        "ticket_id": f"T-{1000 + i}",
        "customer_tier": "enterprise" if i % 2 == 0 else "standard",
        "channel": "chat",
    }

    result = app.invoke(
        make_initial_state(t),
        config={"tags": tags, "metadata": metadata},
    )
    print(f"  ticket T-{1000 + i}  →  {result['category']:9}  tags={tags}")

  ticket T-1000  →  billing    tags=['category:billing', 'non-urgent']
  ticket T-1001  →  technical  tags=['category:technical', 'non-urgent']
  ticket T-1002  →  technical  tags=['category:account', 'urgent']
  ticket T-1003  →  account    tags=['category:account', 'non-urgent']
  ticket T-1004  →  billing    tags=['category:billing', 'non-urgent']


## 6.  Step 3 — Find the planted slow node

If you watched the timings in Step 1, you noticed each run takes ~3–6 seconds. That's not nothing.

**Workflow from the slides:**
1. Open LangSmith → sort runs by Latency descending
2. Open the slowest run → look at the per-span latencies in the run tree
3. Identify which span dominates
4. Drill in — what's different about that span?

Find the answer, then write your diagnosis below.

In [9]:
DIAGNOSIS = {
    "slowest_span": "drafter",
    "root_cause": (
        "The Drafter is instantiated with model='gpt-4o' (see llm_drafter cell above), "
        "while the other agents use 'gpt-4o-mini'. gpt-4o is ~3x slower for similar prompts. "
        "In the trace, the 'drafter' node span takes 2-4s, while every other node is under 0.5s."
    ),
    "fix": (
        "Swap llm_drafter to ChatOpenAI(model='gpt-4o-mini', temperature=0). "
        "For this case study, the cheap model produces drafts of equal quality. "
        "This is the 'model tiering' optimization from the capstone."
    ),
}
print(DIAGNOSIS)


# Apply the fix and re-run to confirm
llm_drafter = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Need to re-bind the draft function since llm_drafter was captured
def draft(state: TriageState) -> dict:
    context = "\n\n".join(f"[{d['id']}] {d['title']}\n{d['text']}"
                            for d in state["retrieved"])
    prompt = DRAFT_PROMPT.format(context=context, ticket=state["ticket"])
    response = llm_drafter.invoke(prompt)
    return {"draft": response.content, "revisions": [response.content]}


# Rebuild graph with the fixed draft function
graph = StateGraph(TriageState)
graph.add_node("classify", classify)
graph.add_node("retrieve", retrieve)
graph.add_node("drafter",  draft)
graph.add_node("qa",       qa)
graph.set_entry_point("classify")
graph.add_edge("classify", "retrieve")
graph.add_edge("retrieve", "drafter")
graph.add_edge("drafter",  "qa")
graph.add_conditional_edges("qa", route_qa,
    {"drafter": "drafter", "END": END})
app = graph.compile()

print("\nRe-running with the fix:")
for t in sample_tickets[:3]:
    t0 = time.time()
    app.invoke(make_initial_state(t),
               config={"tags": ["after-fix"], "metadata": {"experiment": "drafter-tier-down"}})
    print(f"  {time.time() - t0:5.2f}s   {t[:55]}...")

{'slowest_span': 'drafter', 'root_cause': "The Drafter is instantiated with model='gpt-4o' (see llm_drafter cell above), while the other agents use 'gpt-4o-mini'. gpt-4o is ~3x slower for similar prompts. In the trace, the 'drafter' node span takes 2-4s, while every other node is under 0.5s.", 'fix': "Swap llm_drafter to ChatOpenAI(model='gpt-4o-mini', temperature=0). For this case study, the cheap model produces drafts of equal quality. This is the 'model tiering' optimization from the capstone."}

Re-running with the fix:
   2.20s   Hi, my monthly bill is $50 higher than last month. Can ...
   2.72s   I keep getting 429 errors from your API on the Standard...
   5.14s   Forgot my password and the reset email never showed up....


## Wrap up

In this lab you went from print-statement debugging to systematic observability. What you can now do that you couldn't in Module 3:

- **Replay any run** end-to-end — even runs from yesterday
- **Filter by metadata** — "show me all enterprise billing tickets that took > 5s"
- **Compare runs side-by-side** — yesterday vs today, baseline vs new
- **Quantify bottlenecks** — which span actually dominates p95?

**Up next:** Module 5 — Evaluation & Reliability. Traces tell you what happened on individual runs. Eval tells you whether the system as a whole is getting better or worse.